# 冷轧卷序优化建模

这个 notebook 用一个小型、内联的合同卷表展示 OptAgent 如何表达通用冷轧卷序优化问题。重点是建模声明和求解器能力，而不是某一条实际机组的生产系统集成。

业务目标：在一批待冷轧合同卷中，排出一个更平稳的生产顺序，降低宽度跳跃、入口厚度跳跃、目标厚度跳跃、压下率跳跃、钢种/硬度族切换、表面质量块打断、涂油路线切换和交期偏离。

## 1. 表格式示例数据

真实项目里，冷轧排程输入通常来自订单、材料和产能主数据等多张业务表。本示例用 Python 的 `list[dict]` 模拟这些表，既保持轻量，又能看清数据如何进入模型。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import pandas as pd

from optagent import ExternalCallbackContext, ModelBuilder, SolveOptions, TabuConfig, solve


@dataclass(frozen=True)
class ColdRollingCoil:
    # 面向业务的订单标识。生产环境里通常就是计划员熟悉的合同号或生产订单号。
    order_id: str
    # 材料号用于回溯到具体热轧来料卷、酸洗卷或待轧中间卷。
    material_id: str
    # capacity_type 表示通用冷轧产能类型，不绑定某条实际机组。
    capacity_type: str
    # 需求字段：钢种和硬度族切换需要缓冲。grade_family / hardness_family 近似表达轧制窗口、轧制力和延伸率差异。
    grade_family: str
    hardness_family: str
    # 需求字段：厚度、宽度和压下率过渡要平稳。entry_thickness / target_thickness / width / reduction_ratio 进入平滑规则。
    entry_thickness: float
    target_thickness: float
    width: float
    reduction_ratio: float
    weight: float
    # 需求字段：表面质量窗口、涂油路线和交期压力。surface_grade / oiling_route / due_bucket 进入对应规则。
    surface_grade: str
    oiling_route: str
    due_bucket: int
    # 需求字段：交期压力。due_priority 越小越急；target_position 是希望出现的最晚大致位置。
    due_priority: int
    target_position: int
    surface_sensitive: bool = False
    high_reduction: bool = False


### 1.1 产能类型表

产能类型表描述一个通用冷轧能力边界。它不是实际机组清单，而是说明这个模型可以接收哪些能力字段，并在后续扩展为多产能类型分配和排序。这里的 `min_width`、`max_width`、`min_target_thickness` 和 `max_entry_thickness` 对应“规格必须落在可生产能力内”的基础需求，本示例暂不把它们做成硬约束。

In [ ]:
# 产能类型表：描述通用冷轧产能能力。示例暂不把上下限做成硬约束，但这些字段说明模型可扩展的方向。
capacity_table = [
    {"capacity_type": "generic_cold_rolling", "process_name": "通用冷轧产能", "min_width": 850, "max_width": 1650, "min_target_thickness": 0.25, "max_entry_thickness": 4.0},
]

pd.DataFrame(capacity_table)


### 1.2 材料表

材料表记录每卷待轧材料的物理和冶金属性。`entry_thickness`、`width`、`grade_family`、`hardness_family` 和 `weight` 分别支撑厚度平稳、宽度平稳、钢种/硬度切换缓冲和规模展示需求，这些字段会直接影响相邻卷过渡成本。

In [ ]:
# 材料表：一行是一卷待轧材料，保留钢种族、硬度族、入口厚度、宽度和重量等物性。
material_table = [
    {"material_id": "CRM001", "grade_family": "CQ", "hardness_family": "soft", "entry_thickness": 2.10, "width": 1260, "weight": 24.5},
    {"material_id": "CRM002", "grade_family": "CQ", "hardness_family": "soft", "entry_thickness": 2.05, "width": 1250, "weight": 23.8},
    {"material_id": "CRM003", "grade_family": "DQ", "hardness_family": "soft", "entry_thickness": 1.95, "width": 1235, "weight": 22.9},
    {"material_id": "CRM004", "grade_family": "EDDQ", "hardness_family": "soft", "entry_thickness": 2.30, "width": 1420, "weight": 25.2},
    {"material_id": "CRM005", "grade_family": "EDDQ", "hardness_family": "soft", "entry_thickness": 2.25, "width": 1410, "weight": 24.9},
    {"material_id": "CRM006", "grade_family": "HSLA", "hardness_family": "medium", "entry_thickness": 3.20, "width": 1520, "weight": 28.5},
    {"material_id": "CRM007", "grade_family": "HSLA", "hardness_family": "medium", "entry_thickness": 3.10, "width": 1505, "weight": 27.9},
    {"material_id": "CRM008", "grade_family": "DP", "hardness_family": "hard", "entry_thickness": 2.60, "width": 1180, "weight": 23.1},
    {"material_id": "CRM009", "grade_family": "DP", "hardness_family": "hard", "entry_thickness": 2.55, "width": 1170, "weight": 22.6},
    {"material_id": "CRM010", "grade_family": "CQ", "hardness_family": "soft", "entry_thickness": 2.00, "width": 1275, "weight": 23.4},
]

pd.DataFrame(material_table)


### 1.3 订单表

订单表把材料和客户/工艺要求关联起来。`target_thickness` 支撑目标厚度平稳需求，`surface_grade` 和 `surface_sensitive` 支撑表面质量窗口保护需求，`oiling_route` 支撑涂油路线连续需求，`due_bucket`、`due_priority`、`target_position` 支撑交期压力需求。这些字段不是展示字段，它们会进入目标函数，影响最终卷序。

In [ ]:
# 订单表：一行是一张待排生产订单，表达目标厚度、表面等级、涂油路线和交期压力。
# 需求字段：target_thickness -> 目标厚度平稳；surface_grade/surface_sensitive -> 表面质量窗口；oiling_route -> 涂油连续；due_* -> 交期压力。
order_table = [
    {"order_id": "CRO1001", "material_id": "CRM001", "capacity_type": "generic_cold_rolling", "target_thickness": 0.82, "surface_grade": "standard", "surface_sensitive": False, "oiling_route": "light_oil", "due_bucket": 2, "due_priority": 2, "target_position": 4},
    {"order_id": "CRO1002", "material_id": "CRM002", "capacity_type": "generic_cold_rolling", "target_thickness": 0.80, "surface_grade": "standard", "surface_sensitive": False, "oiling_route": "light_oil", "due_bucket": 2, "due_priority": 2, "target_position": 5},
    {"order_id": "CRO1003", "material_id": "CRM003", "capacity_type": "generic_cold_rolling", "target_thickness": 0.70, "surface_grade": "standard", "surface_sensitive": False, "oiling_route": "light_oil", "due_bucket": 1, "due_priority": 1, "target_position": 3},
    {"order_id": "CRO1004", "material_id": "CRM004", "capacity_type": "generic_cold_rolling", "target_thickness": 0.65, "surface_grade": "exposed", "surface_sensitive": True, "oiling_route": "anti_rust_oil", "due_bucket": 3, "due_priority": 3, "target_position": 7},
    {"order_id": "CRO1005", "material_id": "CRM005", "capacity_type": "generic_cold_rolling", "target_thickness": 0.66, "surface_grade": "exposed", "surface_sensitive": True, "oiling_route": "anti_rust_oil", "due_bucket": 3, "due_priority": 3, "target_position": 8},
    {"order_id": "CRO1006", "material_id": "CRM006", "capacity_type": "generic_cold_rolling", "target_thickness": 1.20, "surface_grade": "structural", "surface_sensitive": False, "oiling_route": "dry", "due_bucket": 4, "due_priority": 4, "target_position": 9},
    {"order_id": "CRO1007", "material_id": "CRM007", "capacity_type": "generic_cold_rolling", "target_thickness": 1.18, "surface_grade": "structural", "surface_sensitive": False, "oiling_route": "dry", "due_bucket": 4, "due_priority": 4, "target_position": 10},
    {"order_id": "CRO1008", "material_id": "CRM008", "capacity_type": "generic_cold_rolling", "target_thickness": 0.55, "surface_grade": "high_strength", "surface_sensitive": True, "oiling_route": "heavy_oil", "due_bucket": 5, "due_priority": 5, "target_position": 10},
    {"order_id": "CRO1009", "material_id": "CRM009", "capacity_type": "generic_cold_rolling", "target_thickness": 0.56, "surface_grade": "high_strength", "surface_sensitive": True, "oiling_route": "heavy_oil", "due_bucket": 5, "due_priority": 5, "target_position": 10},
    {"order_id": "CRO1010", "material_id": "CRM010", "capacity_type": "generic_cold_rolling", "target_thickness": 0.78, "surface_grade": "standard", "surface_sensitive": False, "oiling_route": "light_oil", "due_bucket": 1, "due_priority": 1, "target_position": 2},
]

pd.DataFrame(order_table)


### 1.4 表连接为模型输入

求解器最终需要的是“待排序对象”和一个可复现的基线排列。下面这一步把订单表和材料表连接起来，生成模型使用的 `ColdRollingCoil` 对象列表，并用订单表中的行顺序作为 `baseline_sequence`。

In [ ]:
def build_coils_from_tables() -> tuple[list[ColdRollingCoil], list[int]]:
    # 用 material_id 把材料物性补到订单上；这一步相当于 APS/MES 前处理中的轻量 join。
    material_by_id = {row["material_id"]: row for row in material_table}
    coils: list[ColdRollingCoil] = []
    for order in order_table:
        material = material_by_id[order["material_id"]]
        reduction_ratio = (material["entry_thickness"] - order["target_thickness"]) / material["entry_thickness"]
        coils.append(ColdRollingCoil(
            order_id=order["order_id"],
            material_id=order["material_id"],
            capacity_type=order["capacity_type"],
            grade_family=material["grade_family"],
            hardness_family=material["hardness_family"],
            entry_thickness=material["entry_thickness"],
            target_thickness=order["target_thickness"],
            width=material["width"],
            reduction_ratio=round(reduction_ratio, 4),
            weight=material["weight"],
            surface_grade=order["surface_grade"],
            oiling_route=order["oiling_route"],
            due_bucket=order["due_bucket"],
            due_priority=order["due_priority"],
            target_position=order["target_position"],
            surface_sensitive=order["surface_sensitive"],
            high_reduction=reduction_ratio >= 0.68,
        ))

    # 这里直接使用订单表行顺序作为可复现的基线顺序。
    baseline_sequence = list(range(len(coils)))
    return coils, baseline_sequence


coils, baseline_sequence = build_coils_from_tables()

# 展示业务输入表和 join 后的模型输入。notebook 输出中可以逐表查看，交互感比单一列表更接近真实系统。
model_input_df = pd.DataFrame([coil.__dict__ for coil in coils])
model_input_df


## 2. 需求声明位置对照

下表把 README 中的业务需求映射到模型里的声明位置。读 notebook 时可以先看字段，再看规则成本，确认每个需求都有明确字段来源和评分规则。

In [ ]:
requirement_mapping = [
    {"business_requirement": "厚度和压下率过渡要平稳", "declared_in_table": "材料表 + 订单表", "declared_in_fields": "entry_thickness, target_thickness, reduction_ratio", "declared_in_rules": "smooth_entry_thickness, smooth_target_thickness, smooth_reduction_ratio"},
    {"business_requirement": "宽度跳跃要受控，降低板形和辊形风险", "declared_in_table": "材料表", "declared_in_fields": "width", "declared_in_rules": "smooth_width, roll_profile_risk"},
    {"business_requirement": "钢种和硬度族切换需要缓冲", "declared_in_table": "材料表", "declared_in_fields": "grade_family, hardness_family", "declared_in_rules": "grade_family_change, hardness_family_change"},
    {"business_requirement": "表面质量窗口需要保护", "declared_in_table": "订单表", "declared_in_fields": "surface_grade, surface_sensitive", "declared_in_rules": "surface_window_break"},
    {"business_requirement": "涂油和后续路线切换要减少", "declared_in_table": "订单表", "declared_in_fields": "oiling_route", "declared_in_rules": "oiling_route_change"},
    {"business_requirement": "交期压力要体现，同时避免急单过晚", "declared_in_table": "订单表", "declared_in_fields": "due_bucket, due_priority, target_position", "declared_in_rules": "due_bucket_spread, due_position_risk"},
]

pd.DataFrame(requirement_mapping)


## 3. 业务规则评分

下面的目标函数保持在业务语言层：相邻卷的宽度、入口厚度、目标厚度和压下率越平滑越好；钢种族、硬度族、表面质量块和涂油路线尽量连续；宽度大幅上跳时增加换辊风险；交期则同时考虑相邻窗口混排和急单位置过晚两类风险。这个函数就是传给 OptAgent 的黑箱评分函数。

In [ ]:
def transition_cost(prev: ColdRollingCoil, curr: ColdRollingCoil) -> dict[str, float]:
    # 每一对相邻卷都会产生一组成本。这样评分函数就贴近计划员的表达：
    # “把 curr 接在 prev 后面轧制，到底有多不顺？”
    width_jump = abs(prev.width - curr.width)
    entry_jump = abs(prev.entry_thickness - curr.entry_thickness)
    target_jump = abs(prev.target_thickness - curr.target_thickness)
    reduction_jump = abs(prev.reduction_ratio - curr.reduction_ratio)
    width_up_jump = max(0.0, curr.width - prev.width)
    due_jump = abs(prev.due_bucket - curr.due_bucket)

    return {
        # 需求：厚度和压下率跳变影响轧制稳定。
        "smooth_width": width_jump * 0.45,
        "smooth_entry_thickness": entry_jump * 260.0,
        "smooth_target_thickness": target_jump * 520.0,
        "smooth_reduction_ratio": reduction_jump * 950.0,
        # 需求：钢种和硬度族切换需要缓冲。
        "grade_family_change": 180.0 if prev.grade_family != curr.grade_family else 0.0,
        "hardness_family_change": 220.0 if prev.hardness_family != curr.hardness_family else 0.0,
        # 需求：表面质量窗口、涂油路线和高压下率材料需要尽量连续。
        "surface_window_break": 260.0 if prev.surface_sensitive != curr.surface_sensitive else 0.0,
        "high_reduction_change": 150.0 if prev.high_reduction != curr.high_reduction else 0.0,
        "oiling_route_change": 140.0 if prev.oiling_route != curr.oiling_route else 0.0,
        # 需求：宽度大幅上跳会增加辊形/板形风险。
        "roll_profile_risk": max(0.0, width_up_jump - 40.0) * 2.5,
        # 需求：交期窗口不宜剧烈混排。
        "due_bucket_spread": max(0, due_jump - 1) * 80.0,
    }


def sequence_breakdown(sequence: list[int]) -> dict[str, float]:
    # 每类规则成本单独保留，便于解释最终结果。计划员可以看出改进来自规格更平滑、
    # 表面质量块减少打断，还是其他规则之间的取舍。
    costs = {
        "smooth_width": 0.0,
        "smooth_entry_thickness": 0.0,
        "smooth_target_thickness": 0.0,
        "smooth_reduction_ratio": 0.0,
        "grade_family_change": 0.0,
        "hardness_family_change": 0.0,
        "surface_window_break": 0.0,
        "high_reduction_change": 0.0,
        "oiling_route_change": 0.0,
        "roll_profile_risk": 0.0,
        "due_bucket_spread": 0.0,
        "due_position_risk": 0.0,
    }
    # 沿着候选序列逐对累加相邻卷过渡成本。
    for left, right in zip(sequence, sequence[1:]):
        for name, value in transition_cost(coils[left], coils[right]).items():
            costs[name] += value

    # 需求：交期压力和工艺平稳性相互冲突。
    # 相邻 due_bucket 只表达“不要把交期窗口混得太散”；这里再按位置惩罚急单过晚。
    for position, index in enumerate(sequence, start=1):
        coil = coils[index]
        lateness_positions = max(0, position - coil.target_position)
        urgency_weight = max(1, 6 - coil.due_priority)
        costs["due_position_risk"] += lateness_positions * urgency_weight * 95.0
    return {name: round(value, 3) for name, value in costs.items()}


def cold_rolling_sequence_cost(sequence: list[int]) -> float:
    # OptAgent 最小化一个标量目标。这里把标量定义为所有业务成本之和，同时保留 sequence_breakdown
    # 方便求解后做分项诊断。
    return round(sum(sequence_breakdown(sequence).values()), 3)


# baseline 让优化结果更具体：最终报告可以逐条规则对比优化序列和订单表默认顺序。
baseline_costs = sequence_breakdown(baseline_sequence)
baseline_cost = cold_rolling_sequence_cost(baseline_sequence)
baseline_cost, baseline_costs


## 4. OptAgent 建模声明

建模只需要三步：声明一个 `sequence_var`，把业务评分函数接成 `external_call`，再声明最小化目标。求解器负责在排列空间里搜索更好的卷序。

In [ ]:
# 1. 创建 OptAgent 模型。metadata 不是必需项，但当 notebook 扩展成可重复实验或 benchmark 时很有用。
builder = ModelBuilder(metadata={"case": "generic_cold_rolling_coil_sequence"})

# 2. 声明决策变量：合同卷下标的一个排列。default 使用订单表顺序，因此基线和搜索起点都可复现。
sequence = builder.sequence_var(
    size=len(coils),
    default=baseline_sequence,
    name="cold_rolling_coil_sequence",
)


def cold_rolling_rule_cost(ctx: ExternalCallbackContext) -> float:
    order = [int(index) for index in ctx.value(sequence)]
    return cold_rolling_sequence_cost(order)


# 3. 直接接入业务评分函数。external_call 让评分逻辑保持普通 Python 写法，排列搜索交给 OptAgent 处理。
builder.minimize(
    builder.external_call(cold_rolling_rule_cost, name="cold_rolling_rule_cost"),
    name="minimize_cold_rolling_transition_cost",
)
# freeze 会把 builder 中的声明固化成不可变 program，后续交给 solve-first 策略 API。
program = builder.freeze()

program.metadata


## 5. 求解

这里使用 tabu 启发式做快速局部搜索。对示例读者而言，重点是：不需要手写邻域搜索、禁忌表或接受准则，只需要把卷序变量和业务评分交给 OptAgent。

In [ ]:
# TABU 适合做序列修补，因为它会探索局部交换/移动，同时避免马上回到刚访问过的排列。
result = solve(
    program,
    options=SolveOptions(
        strategy=TabuConfig(max_iterations=90, tabu_tenure=9),
        max_iterations=90,
        seed=11,
    ),
)

# UnifiedSolution 按 node_id 存储变量值。这里转成普通 int，方便展示和序列化。
best_sequence = [int(index) for index in result.variable_values[sequence.node_id]]
best_cost = cold_rolling_sequence_cost(best_sequence)
{
    "baseline_cost": baseline_cost,
    "best_cost": best_cost,
    "improvement": round(baseline_cost - best_cost, 3),
    "best_sequence": [coils[index].order_id for index in best_sequence],
}


## 6. 结果解释

输出不只看总分，还要看每类规则的变化。冷轧计划员通常会检查：宽度和厚度是否更平滑、压下率是否少跳变、表面质量敏感材料是否被打散、硬度族和涂油路线是否频繁切换、急单是否被过度后移。

In [ ]:
# 对最优序列重新计算分项规则成本。这张表用于判断优化方案在业务上是否合理。
best_costs = sequence_breakdown(best_sequence)
cost_comparison = []
for name in sorted(baseline_costs):
    # delta 为负表示该规则成本降低；delta 为正表示求解器为了其他目标，在这条规则上付出了更多代价。
    cost_comparison.append({
        "rule": name,
        "baseline": baseline_costs[name],
        "best": best_costs[name],
        "delta": round(best_costs[name] - baseline_costs[name], 3),
    })

pd.DataFrame(cost_comparison)


In [ ]:
def sequence_view(sequence: list[int]) -> list[dict[str, object]]:
    # 把优化得到的下标排列还原成合同卷属性表。这是计划员审批排程前更自然的查看格式。
    rows = []
    for pos, index in enumerate(sequence, start=1):
        coil = coils[index]
        rows.append({
            "pos": pos,
            "order_id": coil.order_id,
            "material_id": coil.material_id,
            "capacity_type": coil.capacity_type,
            "grade_family": coil.grade_family,
            "hardness_family": coil.hardness_family,
            "entry_thickness": coil.entry_thickness,
            "target_thickness": coil.target_thickness,
            "width": coil.width,
            "reduction_ratio": coil.reduction_ratio,
            "surface_grade": coil.surface_grade,
            "oiling_route": coil.oiling_route,
            "due_bucket": coil.due_bucket,
            "due_priority": coil.due_priority,
            "target_position": coil.target_position,
        })
    return rows


pd.DataFrame(sequence_view(best_sequence))


## 7. 完整代码示例

下面的单元把前面的关键步骤合在一起，保留必要注释，适合读者快速复制到一个 Python 文件或空白 notebook 中运行。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

from optagent import ExternalCallbackContext, ModelBuilder, SolveOptions, TabuConfig, solve


@dataclass(frozen=True)
class DemoCoil:
    """一个待冷轧合同卷。示例只保留影响排序的核心字段。"""

    order_id: str
    material_id: str
    grade_family: str
    hardness_family: str
    target_thickness: float
    width: float
    reduction_ratio: float
    surface_sensitive: bool
    oiling_route: str
    due_bucket: int
    due_priority: int
    target_position: int


# 订单表和材料表是两个常见业务输入。这里用内联数据代替数据库，方便示例自包含。
demo_material_table = [
    {"material_id": "CRM001", "grade_family": "CQ", "hardness_family": "soft", "entry_thickness": 2.10, "width": 1260},
    {"material_id": "CRM002", "grade_family": "DQ", "hardness_family": "soft", "entry_thickness": 1.95, "width": 1235},
    {"material_id": "CRM003", "grade_family": "EDDQ", "hardness_family": "soft", "entry_thickness": 2.30, "width": 1420},
    {"material_id": "CRM004", "grade_family": "HSLA", "hardness_family": "medium", "entry_thickness": 3.20, "width": 1520},
]
demo_order_table = [
    {"order_id": "CRO1001", "material_id": "CRM001", "target_thickness": 0.82, "surface_sensitive": False, "oiling_route": "light_oil", "due_bucket": 2, "due_priority": 2, "target_position": 2},
    {"order_id": "CRO1003", "material_id": "CRM003", "target_thickness": 0.65, "surface_sensitive": True, "oiling_route": "anti_rust_oil", "due_bucket": 3, "due_priority": 3, "target_position": 4},
    {"order_id": "CRO1004", "material_id": "CRM004", "target_thickness": 1.20, "surface_sensitive": False, "oiling_route": "dry", "due_bucket": 4, "due_priority": 4, "target_position": 4},
    {"order_id": "CRO1002", "material_id": "CRM002", "target_thickness": 0.70, "surface_sensitive": False, "oiling_route": "light_oil", "due_bucket": 1, "due_priority": 1, "target_position": 1},
]


def build_demo_coils() -> tuple[list[DemoCoil], list[int]]:
    materials = {row["material_id"]: row for row in demo_material_table}
    demo_coils: list[DemoCoil] = []
    for order in demo_order_table:
        material = materials[order["material_id"]]
        reduction_ratio = (material["entry_thickness"] - order["target_thickness"]) / material["entry_thickness"]
        demo_coils.append(DemoCoil(
            order_id=order["order_id"],
            material_id=order["material_id"],
            grade_family=material["grade_family"],
            hardness_family=material["hardness_family"],
            target_thickness=order["target_thickness"],
            width=material["width"],
            reduction_ratio=round(reduction_ratio, 4),
            surface_sensitive=order["surface_sensitive"],
            oiling_route=order["oiling_route"],
            due_bucket=order["due_bucket"],
            due_priority=order["due_priority"],
            target_position=order["target_position"],
        ))
    return demo_coils, list(range(len(demo_coils)))


demo_coils, demo_baseline_sequence = build_demo_coils()


def demo_sequence_cost(sequence: list[int]) -> float:
    total = 0.0
    for left, right in zip(sequence, sequence[1:]):
        prev = demo_coils[left]
        curr = demo_coils[right]
        total += abs(prev.width - curr.width) * 0.45
        total += abs(prev.target_thickness - curr.target_thickness) * 520.0
        total += abs(prev.reduction_ratio - curr.reduction_ratio) * 950.0
        total += 180.0 if prev.grade_family != curr.grade_family else 0.0
        total += 220.0 if prev.hardness_family != curr.hardness_family else 0.0
        total += 260.0 if prev.surface_sensitive != curr.surface_sensitive else 0.0
        total += 140.0 if prev.oiling_route != curr.oiling_route else 0.0
        total += max(0, abs(prev.due_bucket - curr.due_bucket) - 1) * 80.0
    for position, index in enumerate(sequence, start=1):
        coil = demo_coils[index]
        total += max(0, position - coil.target_position) * max(1, 6 - coil.due_priority) * 95.0
    return round(total, 3)


demo_builder = ModelBuilder(metadata={"case": "compact_cold_rolling_demo"})
demo_sequence = demo_builder.sequence_var(size=len(demo_coils), default=demo_baseline_sequence, name="demo_cold_rolling_sequence")


def demo_cold_rolling_cost(ctx: ExternalCallbackContext) -> float:
    return demo_sequence_cost([int(index) for index in ctx.value(demo_sequence)])


demo_builder.minimize(
    demo_builder.external_call(demo_cold_rolling_cost, name="demo_cold_rolling_cost"),
    name="minimize_demo_cold_rolling_cost",
)
demo_program = demo_builder.freeze()

demo_result = solve(
    demo_program,
    options=SolveOptions(strategy=TabuConfig(max_iterations=40, tabu_tenure=6), max_iterations=40, seed=3),
)
demo_best_sequence = [int(index) for index in demo_result.variable_values[demo_sequence.node_id]]
{
    "baseline_cost": demo_sequence_cost(demo_baseline_sequence),
    "best_cost": demo_sequence_cost(demo_best_sequence),
    "best_sequence": [demo_coils[index].order_id for index in demo_best_sequence],
}
